# GrihaMool: Har Ghar Ki Sahi Keemat — House Price Dataset EDA

This notebook profiles the bundled dataset before model training. Run from the repository root or from this notebook; the path detection below supports both. **The included dataset is synthetic demonstration data**, so patterns and metrics are not evidence about actual real-estate markets.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

candidates = [Path("data/properties.csv"), Path("../data/properties.csv")]
DATA_PATH = next((path for path in candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find data/properties.csv. Open the repository root or notebooks folder.")
df = pd.read_csv(DATA_PATH)
df.head()

## 1. Dataset overview

In [ ]:
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
display(df.sample(min(5, len(df)), random_state=42))
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

## 2. Data quality checks

In [ ]:
quality = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=False),
})
display(quality)
print("Duplicate rows:", int(df.duplicated().sum()))
if "property_id" in df.columns:
    print("Duplicate property IDs:", int(df["property_id"].duplicated().sum()))

## 3. Numeric summary

In [ ]:
numeric_cols = [c for c in ["price_inr", "area_sqft", "bedrooms", "bathrooms", "parking", "age_years"] if c in df.columns]
display(df[numeric_cols].describe().T)

## 4. Price distribution

In [ ]:
if "price_inr" in df:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    df["price_inr"].dropna().hist(bins=30, ax=ax)
    ax.set_title("Distribution of property prices (INR)")
    ax.set_xlabel("Price (INR)")
    ax.set_ylabel("Number of properties")
    fig.tight_layout()
    plt.show()

## 5. Area versus price

In [ ]:
if {"area_sqft", "price_inr"}.issubset(df.columns):
    plot_df = df[["area_sqft", "price_inr"]].dropna()
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(plot_df["area_sqft"], plot_df["price_inr"], alpha=0.45)
    ax.set_title("Property area versus price")
    ax.set_xlabel("Area (sq ft)")
    ax.set_ylabel("Price (INR)")
    fig.tight_layout()
    plt.show()

## 6. Price by bedroom count

In [ ]:
if {"bedrooms", "price_inr"}.issubset(df.columns):
    plot_df = df[["bedrooms", "price_inr"]].dropna()
    groups = [group["price_inr"].values for _, group in plot_df.groupby("bedrooms")]
    labels = [str(value) for value, _ in plot_df.groupby("bedrooms")]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.boxplot(groups, labels=labels, showfliers=False)
    ax.set_title("Price distribution by bedroom count")
    ax.set_xlabel("Bedrooms")
    ax.set_ylabel("Price (INR)")
    fig.tight_layout()
    plt.show()

## 7. Location analysis

In [ ]:
if {"city", "price_inr"}.issubset(df.columns):
    city_summary = df.groupby("city")["price_inr"].agg(["count", "median", "mean"]).sort_values("median", ascending=False)
    display(city_summary)
    fig, ax = plt.subplots(figsize=(10, 5))
    city_summary["median"].sort_values().plot(kind="barh", ax=ax)
    ax.set_title("Median listed price by city (synthetic data)")
    ax.set_xlabel("Median price (INR)")
    ax.set_ylabel("City")
    fig.tight_layout()
    plt.show()

## 8. Conclusions and next steps

Use the outputs above to write findings in `reports/EDA_report.md`. Before training, agree with the team on the target column (`price_inr`) and feature columns used in `src/train_model.py`. Review outliers and implausible values rather than removing rows automatically. Document all cleaning choices.

**Limitation:** This repository currently includes synthetic data for a runnable demo. Replace it with a legally usable, verified dataset for any real-world analysis or valuation claim.